# Imports & Setup

In [23]:
from dataclasses import dataclass
from enum import Enum

# Data Structures

In [24]:
class Operation(Enum):
    """Type of edit operation between source and target words."""

    KEEP = "k"
    REPLACE = "r"
    INSERT = "i"
    DELETE = "d"
    MERGE = "m"
    SPLIT = "s"


@dataclass
class WordAlignment:
    """Represents an alignment between source and target word spans.

    Stores the alignment between a span of source words and a span of target
    words, along with the edit operation.
    """

    source_start: int
    source_end: int

    target_start: int
    target_end: int

    operation: Operation


@dataclass
class BackPointer:
    """Represents a back pointer in the DP table.

    Stores the operation and previous indices for backtracking through the
    dynamic programming table.
    """

    operation: Operation

    prev_i: int
    prev_j: int

# 1. Alignement

In [31]:
class Aligner:
    """Aligns two lists of words using dynamic programming."""

    INSERT_DELETE_COST = 1
    REPLACE_COST = 2
    MERGE_COST = 1
    SPLIT_COST = 1

    def align_words(self, source: list[str], target: list[str]) -> list[WordAlignment]:
        """Aligns two lists of words and returns a list of WordAlignment."""
        # TODO: check the equality based on normalized text rather than edit
        # distance since we only care about word alignment and similarity
        # between words, not the actual edit distance (e.g., "كتاب" and
        # "مكتبة" can have high edit distance but should be aligned).
        # After alignment, check character-level edit distance for edit tag.

        dp, parent = self._build_dp(source, target)
        return self._backtrack(source, target, parent)

    def align_characters(self, source: str, target: str) -> list[WordAlignment]:
        """Aligns two strings at character level."""
        source_chars = list(source)
        target_chars = list(target)

        dp, parent = self._build_dp(source_chars, target_chars)
        return self._backtrack(source_chars, target_chars, parent)

    def levenshtein(self, a: str, b: str) -> int:
        """Calculate the Levenshtein distance between two strings."""
        n = len(a)
        m = len(b)

        if n == 0:
            return m
        if m == 0:
            return n

        dp = [[0] * (m + 1) for _ in range(n + 1)]

        for i in range(n + 1):
            dp[i][0] = i

        for j in range(m + 1):
            dp[0][j] = j

        for i in range(1, n + 1):
            for j in range(1, m + 1):
                dp[i][j] = 1 + min(dp[i - 1][j], dp[i][j - 1], dp[i - 1][j - 1])

                if a[i - 1] == b[j - 1]:
                    dp[i][j] = dp[i - 1][j - 1]

        return dp[n][m]

    def _word_cost(self, source_word: str, target_word: str) -> float:
        """Computes the cost of replacing source_word with target_word."""
        if source_word == target_word:
            return 0.0

        distance = self.levenshtein(source_word, target_word)
        return distance

    def _merge_cost(self, left: str, right: str, target: str) -> float:
        """Computes the cost of merging left and right into target."""
        merged = left + right
        return self._word_cost(merged, target)

    def _split_cost(self, source: str, left: str, right: str) -> float:
        """Computes the cost of splitting source into left and right."""
        split_target = left + right
        return self._word_cost(source, split_target)

    def _build_dp(self, source: list[str], target: list[str]):
        """Builds the DP table and parent pointers for the given source and target."""
        n = len(source)
        m = len(target)

        dp = [[float("inf")] * (m + 1) for _ in range(n + 1)]
        parent = [[None] * (m + 1) for _ in range(n + 1)]
        dp[0][0] = 0

        for i in range(1, n + 1):
            dp[i][0] = dp[i - 1][0] + self.INSERT_DELETE_COST * len(source[i - 1])
            parent[i][0] = BackPointer(Operation.DELETE, i - 1, 0)

        for j in range(1, m + 1):
            dp[0][j] = dp[0][j - 1] + self.INSERT_DELETE_COST * len(target[j - 1])
            parent[0][j] = BackPointer(Operation.INSERT, 0, j - 1)

        for i in range(1, n + 1):
            for j in range(1, m + 1):
                # KEEP && REPLACE
                replace_cost = dp[i - 1][j - 1] + self._word_cost(
                    source[i - 1], target[j - 1]
                )
                if replace_cost < dp[i][j]:
                    dp[i][j] = replace_cost

                    op = (
                        Operation.KEEP
                        if source[i - 1] == target[j - 1]
                        else Operation.REPLACE
                    )
                    parent[i][j] = BackPointer(op, i - 1, j - 1)

                # DELETE
                delete_cost = dp[i - 1][j] + self.INSERT_DELETE_COST * len(
                    source[i - 1]
                )
                if delete_cost < dp[i][j]:
                    dp[i][j] = delete_cost

                    parent[i][j] = BackPointer(Operation.DELETE, i - 1, j)

                # INSERT
                insert_cost = dp[i][j - 1] + self.INSERT_DELETE_COST * len(
                    target[j - 1]
                )
                if insert_cost < dp[i][j]:
                    dp[i][j] = insert_cost

                    parent[i][j] = BackPointer(Operation.INSERT, i, j - 1)

                # MERGE TODO: make it iterative for multiple merges
                if i >= 2:
                    merge_cost = dp[i - 2][j - 1] + self._merge_cost(
                        source[i - 2], source[i - 1], target[j - 1]
                    )

                    if merge_cost < dp[i][j]:
                        dp[i][j] = merge_cost

                        parent[i][j] = BackPointer(Operation.MERGE, i - 2, j - 1)

                # SPLIT TODO: make it iterative for multiple splits
                if j >= 2:
                    split_cost = dp[i - 1][j - 2] + self._split_cost(
                        source[i - 1],
                        target[j - 2],
                        target[j - 1],
                    )

                    if split_cost < dp[i][j]:
                        dp[i][j] = split_cost
                        parent[i][j] = BackPointer(Operation.SPLIT, i - 1, j - 2)
        return dp, parent

    def _backtrack(self, source, target, parent):
        """Backtrack through parent pointers to construct alignments.

        Backtracks through the parent pointers to construct the list of
        WordAlignment objects.
        """
        i = len(source)
        j = len(target)

        alignments = []
        while i > 0 or j > 0:
            ptr = parent[i][j]
            op = ptr.operation

            if op in (Operation.KEEP, Operation.REPLACE):
                alignments.append(
                    WordAlignment(
                        source_start=i - 1,
                        source_end=i - 1,
                        target_start=j - 1,
                        target_end=j - 1,
                        operation=op,
                    )
                )

            elif op == Operation.MERGE:
                alignments.append(
                    WordAlignment(
                        source_start=i - 2,
                        source_end=i - 1,
                        target_start=j - 1,
                        target_end=j - 1,
                        operation=op,
                    )
                )

            elif op == Operation.SPLIT:
                alignments.append(
                    WordAlignment(
                        source_start=i - 1,
                        source_end=i - 1,
                        target_start=j - 2,
                        target_end=j - 1,
                        operation=op,
                    )
                )

            elif op == Operation.DELETE:
                alignments.append(
                    WordAlignment(
                        source_start=i - 1,
                        source_end=i - 1,
                        target_start=j,
                        target_end=j - 1,
                        operation=op,
                    )
                )

            elif op == Operation.INSERT:
                alignments.append(
                    WordAlignment(
                        source_start=i,
                        source_end=i - 1,
                        target_start=j - 1,
                        target_end=j - 1,
                        operation=op,
                    )
                )

            i = ptr.prev_i
            j = ptr.prev_j

        alignments.reverse()
        return alignments

trying some distance and similarity metrics to see how they perform on the task of aligning two sentences and extracting the edits between them.

In [ ]:
from difflib import SequenceMatcher


def similarity(a: str, b: str) -> float:
    """Calculate the similarity ratio between two strings using SequenceMatcher."""
    return SequenceMatcher(
        None,
        a,
        b,
    ).ratio()


def common_prefix_ratio(a, b):
    """Calculate the ratio of common prefix length to max length."""
    count = 0

    for x, y in zip(a, b, strict=False):
        if x != y:
            break

        count += 1

    return count / max(len(a), len(b))

In [38]:
source = ["I", "am", "lo", "ve", "Pythin"]
target = ["I", "love", "Python"]

aligner = Aligner()
alignments = aligner.align_words(source, target)
alignments = aligner.align_characters("li ve", "love")

for alignment in alignments:
    print(alignment)

WordAlignment(source_start=0, source_end=0, target_start=0, target_end=0, operation=<Operation.KEEP: 'k'>)
WordAlignment(source_start=1, source_end=1, target_start=1, target_end=0, operation=<Operation.DELETE: 'd'>)
WordAlignment(source_start=2, source_end=2, target_start=1, target_end=1, operation=<Operation.REPLACE: 'r'>)
WordAlignment(source_start=3, source_end=3, target_start=2, target_end=2, operation=<Operation.KEEP: 'k'>)
WordAlignment(source_start=4, source_end=4, target_start=3, target_end=3, operation=<Operation.KEEP: 'k'>)


# Edit Compression